## GRUPO X
José Pedro Rodrigues - up202006455

Pedro Leite - up199905053

Sandra Silva - up201403358

# Variant Identifier

> Note: For the code tu run faster, change the runtime of this notebook to have a GPU.
On the top tab select "Runtime > Change runtime type > GPU". This change will make the code run in seconds rather than hours...

In an era where chat bots are becoming ubiquitous tools, a significant concern arises: how can we ensure these tools benefit all societies? One major hurdle in achieving this inclusivity is the scarcity of language variant-specific models. For example, while Portuguese is not considered a low-resource language, the majority of available content is in Brazilian Portuguese. Consequently, a language model trained in a Portuguese corpus (without any concern regarding its variants) is likely to exhibit a bias towards producing text in Brazilian Portuguese. What are the implications of such a bias? Countries like Portugal could find themselves at a disadvantage, particularly in deploying language model-based systems in critical areas such as healthcare and judiciary, where the distinct nuances of Portuguese are of great importance.

With this context in mind, your task in this assignment is to develop a model capable of classifying texts as either European Portuguese or Brazilian Portuguese. Below is the quick start code to guide you.

In [ ]:
!pip3 install --quiet datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 12.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.1 MB/s eta 0:00:00


In [ ]:

import pandas
from datasets import load_dataset
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm
from torchtext.vocab import FastText
from transformers import BertModel, BertTokenizer
import pandas as pd
from sklearn.metrics import classification_report


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device {DEVICE} available.")


Device cuda available.


#Datasets

This code bellow allows to extend the original dataset from HuggingFace (cc4051/pt_vid’) using 15 different sources. To import a specific dataset/subdataset the 'import' list should be turn to 1 (if 0 is choosen, no import is considered). Multiple dataset can be imported. To import the github dataset the boolean github_dataset_import is True.

The max_rows parameter paramter limits the size of the imported dataset. By default, it is set to 12717 (equal to the original dataset), a higher value can be considered.

In [ ]:
dataset_import=True
github_dataset_import=True

dict_dataset=dict()

df_gtrain=pd.DataFrame()
df_gtest=pd.DataFrame()

max_rows=12717

if dataset_import:

    # Hugging Face

    dict_dataset[0]={'name':'cc4051',
                     'subset':None,
                     'testortrain': ['train'],
                     'import':[1],
                     'url':'cc4051/pt_vid'}


    dict_dataset[1]={'name':'frmt',
                     'subset':None,
                     'testortrain': ['test'],
                     'import': [0],
                     'url':'LCA-PORVID/frmt'}
#Correr web dict_dataset
    dict_dataset[2]={'name':'portuguese_vid',
                     'subset':['journalistic','legal','literature','politics','social_media','web'],
                     'testortrain':['test','test','test','test','test','train'],
                     'import': [0,0,0,0,0,0],
                     'url':'LCA-PORVID/portuguese_vid'}

    dict_dataset[3]={'name':'delexicalized_n_grams',
                     'subset':['journalistic','legal','literature','politics','social_media','web'],
                     'testortrain':['train','train','train','train','train','train'],
                     'import': [0, 0, 0, 0, 0, 0],
                     'url':'LCA-PORVID/delexicalized_n_grams'}

    #Test - Original

    local_dataset = load_dataset('cc4051/pt_vid')['test']

    local_df = pd.DataFrame(local_dataset)

    df_gtest = pd.concat([df_gtest,local_df], ignore_index=True)

    # Train

    for key in dict_dataset.keys():

        for index,boo_imp in enumerate(dict_dataset[key]['import']):

            if boo_imp == 1:

                t2t = dict_dataset[key]["testortrain"][index]

                if dict_dataset[key]["subset"]==None:

                    local_dataset = load_dataset(dict_dataset[key]['url'])
                    local_df_train = pd.DataFrame(local_dataset[t2t])

                else:

                    subset = dict_dataset[key]["subset"][index]
                    local_dataset = load_dataset(dict_dataset[key]['url'],subset)
                    local_df_train = pd.DataFrame(local_dataset[t2t])

                # Shuffle the entire DataFrame
                local_df_train = local_df_train.sample(frac=1).reset_index(drop=True)

                # Determine the number of rows to sample (whichever is smaller: 10,000 or the number of available rows)
                n_rows = min(max_rows, len(local_df_train))

                # Limit the DataFrame to the determined number of rows
                local_df_train = local_df_train.sample(n=n_rows, random_state=1).reset_index(drop=True)

                df_gtrain = pd.concat([df_gtrain, local_df_train], ignore_index=True)

    # Github

    if github_dataset_import:

        # Github DSL - TL (https://github.com/LanguageTechnologyLab/DSL-TL/tree/main/DSL-TL-Corpus/PT-DSL-TL)
        column_names = ['text', 'label']
        df_train = pd.read_csv("PT_train.tsv", sep='\t', names=column_names)
        df_train['label'] = df_train['label'].replace('PT', 'PT-PT')
        df_train['label'] = df_train['label'].replace('PT-PT', 0)
        df_train['label'] = df_train['label'].replace('PT-BR', 1)
        df_test = pd.read_csv("PT_dev.tsv", sep='\t', names=column_names)
        df_test['label'] = df_test['label'].replace('PT', 'PT-PT')
        df_test['label'] = df_test['label'].replace('PT-PT', 0)
        df_test['label'] = df_test['label'].replace('PT-BR', 1)

        # Concatenar

        df_train = pd.concat([df_gtrain, df_train], ignore_index=True)

    dataset = load_dataset("cc4051/pt_vid")
    dataset['train'] = datasets.Dataset.from_pandas(df_gtrain)

    print("Number of training examples: ", len(dataset["train"]))
    print("Number of testing examples: ", len(dataset["test"]))

else:

    dataset = load_dataset("cc4051/pt_vid")
    print(dataset)



In [ ]:
label_map = {
    0: "PT-PT",
    1: "PT-BR",
}

n_classes = len(label_map)

In [ ]:
dataset

# The modified LSTM model


A LSTM (Long Short-Term Memory) RNN model was used with an initial embedding layer corresponding to the encoder (with the size of the vocab) and an output layer with two neurons (classification in PT-PT or PT-BZ).

Comparing with the original code we created a validation dataset, annotating their scores and stopping the learning process when the validation drops by a number monitored through the patience parameter (80% for training and 20% for validation).

The model allows to introduce a list with the specific values for each hidden layer.



In [ ]:
corpus = "".join(dataset["train"]["text"])

# we will define our vocab to be composed of
vocab = list(set(corpus.lower()))

# Special padding token
pad_token = "<pad>"
vocab.append(pad_token)

# Special unknown token
unk_token = "<unk>"
vocab.append(unk_token)

n_tokens = len(vocab)

print("Number of tokens in the vocab: ", n_tokens)
tkn2id = {v: k for k, v in enumerate(vocab)}

pad_token_id = tkn2id[pad_token]
unk_token_id = tkn2id[unk_token]


def tokenize(text):
    return [tkn2id[c] if c in tkn2id else unk_token_id for c in text.lower()]


In [ ]:
class Model(nn.Module):

    def __init__(
        self,
        vocab_size: int,
        hidden_sizes: list,
        n_classes: int,
        n_layers: int
    ):

        super(Model, self).__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            hidden_sizes[0],
            padding_idx=pad_token_id
        )

        self.lstm_layers = nn.ModuleList()
        for i in range(len(hidden_sizes) - 1):
            self.lstm_layers.append(
                nn.LSTM(
                    input_size=hidden_sizes[i],
                    hidden_size=hidden_sizes[i + 1],
                    batch_first=True,
                )
            )

        self.fc = nn.Linear(hidden_sizes[-1], n_classes)

    def forward(self, x):
        x = self.embedding(x)
        for lstm in self.lstm_layers:
            x, _ = lstm(x)
        x = F.elu(x)
        x = x[:, -1, :]  # get the last hidden state
        logits = self.fc(x)
        return logits

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0

    def __call__(self, val_loss):
        if self.best_score is None:
            self.best_score = val_loss
        elif val_loss > self.best_score - self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_loss
            self.counter = 0

In [ ]:
def evaluate(model: nn.Module, val_loader: DataLoader, criterion):

    model.eval()
    val_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            x = batch["tokens"].to(DEVICE)
            y = batch["label"].to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            val_loss += loss.item() * x.size(0)
            y_pred = logits.argmax(dim=1)
            total_correct += (y_pred == y).sum().item()
            total_samples += x.size(0)

    val_loss /= len(val_loader.dataset)
    val_accuracy = total_correct / total_samples

    return val_loss, val_accuracy

In [ ]:
def train(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    n_epochs: int,
    lr: float,
    patience=3
  ):

    optimizer = optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    early_stopping = EarlyStopping(patience=patience)

    for epoch in range(n_epochs):

        model.train()

        n_steps = len(train_loader)

        progress_bar = tqdm(
            total=n_steps, desc=f"Epoch {epoch+1}", position=0, leave=True
        )
        total_loss = 0
        total_correct = 0
        total_samples = 0

        for idx, batch in enumerate(train_loader):
            optimizer.zero_grad()
            x = batch["tokens"].to(DEVICE)
            y = batch["label"].to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            # logging
            total_loss += loss.item()
            y_pred = logits.argmax(dim=1)
            total_correct += (y_pred == y).sum().item()
            progress_bar.set_postfix(
                {
                    "loss": total_loss / (idx + 1),
                    "accuracy": total_correct / (idx + 1) / x.size(0),
                }
            )
            progress_bar.update()

        progress_bar.close()

        with torch.no_grad():  # Evaluation mod

          val_loss, val_accuracy = evaluate(model, val_loader, criterion)
          print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

        early_stopping(val_loss)
        if early_stopping.early_stop:
            print("Early stopping")
            break

In [ ]:
class VIdDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        """The max_length limits the number of tokens in the text.
        Texts that are longer are truncated, and shorter texts are padded."""
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.pad_token_id = pad_token_id

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]
        tokens = self.tokenizer(text)
        if len(tokens) < self.max_length:  # padding
            padding = [self.pad_token_id] * (self.max_length - len(tokens))
            tokens += padding
        else:  # truncate
            tokens = tokens[: self.max_length]

        label = item["label"]
        return {
            "tokens": torch.tensor(tokens),
            "label": torch.tensor(label),
        }

In [ ]:
batch_size=32

model = Model(
    vocab_size=n_tokens,
    hidden_sizes=[200,200],
    n_classes=n_classes,
    n_layers=1
  ).to(DEVICE)

print(model)

# Create datasets
train_dataset = VIdDataset(dataset["train"], tokenize)

# Split dataset: 80% for training, 20% for validation
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
  )

val_loader = DataLoader(
    dataset=val_subset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False
)

train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=100,
    lr=1e-3,
)

def evaluate_(model: nn.Module, testset: Dataset):
    test_loader = DataLoader(testset, batch_size=batch_size, shuffle=False)

    model.eval()
    y_true, y_pred = [], []
    for batch in test_loader:
        x = batch["tokens"].to(DEVICE)
        y_true += batch["label"].tolist()
        logits = model(x)
        y_pred += logits.argmax(dim=1).tolist()

    return y_true, y_pred

testset = VIdDataset(dataset["test"], tokenize)
y_true_test, y_pred_test = evaluate_(model, testset)

print(f"First 10 true labels:{y_true_test[:10]}")
print(f"First 10 pred labels:{y_pred_test[:10]}")


target_names = [label_map[i] for i in range(n_classes)]
report = classification_report(
    y_true_test,
    y_pred_test,
    target_names=target_names,
    digits=4
)
print(report)



In [ ]:
trainset = VIdDataset(dataset["train"], tokenize)
y_true_train, y_pred_train = evaluate_(model, trainset)
report = classification_report(
    y_true_train,
    y_pred_train,
    target_names=target_names,
    digits=4
)
print(report)

# Hyperparameters

## Number of epochs


The number of epochs tested ranged from 1 to 250, changing the hidden_sizes list. The number of layer and hidden size was fixed (2 and 128).

## Learning rate


For this hyperparameter we studied 4 different rates: 1e-1,1e-2,1e-3,1e-4. The best number was 1e-3. The number of layer and hidden size was fixed (2 and 128)

##Batch number


The batch size was fixed to 32 with no considerable difference observed for 16 and 64.

## Activation function

The ReLu and ELU activation functions were evaluated and no significant differences were observed, changing in the F method.



##Number of hidden layers and neurons

The number layers and hidden size was evaluated for the original and extendend dataset.

##Optimiztion function

We also tested the optimization functions influence, considering: AdamW ( original), Adagrad, RMSprop and Adam.
To do this test we fixed the number of epochs equal to 20 and used 4 layers [200,200,200,200].

In [ ]:
def train_optimitazion(
    optimizer_name,
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    n_epochs: int,
    lr: float,
    patience=20
  ):

    if optimizer_name == 'RMSprop':
        optimizer = optim.RMSprop(model.parameters(), lr=lr)
    elif optimizer_name == 'Adagrad':
        optimizer = optim.Adagrad(model.parameters(), lr=lr)
    elif optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name == 'AdamW':
        optimizer = optim.AdamW(model.parameters(), lr=lr)

    start_time = time.time()

    criterion = nn.CrossEntropyLoss()
    early_stopping = EarlyStopping(patience=patience)

    for epoch in range(n_epochs):

        model.train()

        n_steps = len(train_loader)

        progress_bar = tqdm(
            total=n_steps, desc=f"Epoch {epoch+1}", position=0, leave=True
        )
        total_loss = 0
        total_correct = 0
        total_samples = 0

        for idx, batch in enumerate(train_loader):
            optimizer.zero_grad()
            x = batch["tokens"].to(DEVICE)
            y = batch["label"].to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            # logging
            total_loss += loss.item()
            y_pred = logits.argmax(dim=1)
            total_correct += (y_pred == y).sum().item()
            progress_bar.set_postfix(
                {
                    "loss": total_loss / (idx + 1),
                    "accuracy": total_correct / (idx + 1) / x.size(0),
                }
            )
            progress_bar.update()

        progress_bar.close()

        with torch.no_grad():  # Evaluation mod

          val_loss, val_accuracy = evaluate(model, val_loader, criterion)
          print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

        early_stopping(val_loss)
        if early_stopping.early_stop:
            print("Early stopping")
            break

    end_time = time.time()
    elapsed_time = end_time - start_time
    return total_loss, elapsed_time

In [ ]:
batch_size=32

model = Model(
    vocab_size=n_tokens,
    hidden_sizes=[200,200, 200, 200],
    n_classes=n_classes
  ).to(DEVICE)

print(model)

# Create datasets
train_dataset = VIdDataset(dataset["train"], tokenize)

# Split dataset: 80% for training, 20% for validation
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
  )

val_loader = DataLoader(
    dataset=val_subset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False
)

opti_list = ['RMSprop', 'Adagrad', 'Adam','AdamW']
results = []
import time

for i in opti_list:
    torch.cuda.empty_cache()

    train_loss, elapsed_time = train_optimitazion(
        optimizer_name=i,
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        n_epochs=20,
        lr=1e-3
    )

    testset = VIdDataset(dataset["test"], tokenize)
    y_true_test, y_pred_test = evaluate_(model, testset)

    target_names = [label_map[i] for i in range(n_classes)]
    report_test = classification_report(
        y_true_test,
        y_pred_test,
        target_names=target_names,
        digits=4,
        output_dict=True
    )

    trainset = VIdDataset(dataset["train"], tokenize)
    y_true_train, y_pred_train = evaluate_(model, trainset)
    report_train = classification_report(
        y_true_train,
        y_pred_train,
        target_names=target_names,
        digits=4,
        output_dict=True
    )

    results.append({
        "Optimizer": i,
        "Train F1 pt-pt": report_train['PT-PT']['f1-score'],
        "Train F1 pt-br": report_train['PT-BR']['f1-score'],
        "Test F1 pt-pt": report_test['PT-PT']['f1-score'],
        "Test F1 pt-br": report_test['PT-BR']['f1-score'],
        "Time (s)": elapsed_time,
        "Loss": train_loss
    })

df = pd.DataFrame(results)
print(df)

# Pre-trained Components

## Pre-trained Embedings

To improve the results of the model we incorporated in the model word embedings. We tested the Glove and the fastText. Regarding the hyperparameters we used 4 layers [300, 300, 300, 300], and kept the original lerning rate, 1e-3, and the optimization function, AdamW. we tested both Glove and fastText with the original and extended datasets.

In [ ]:
class Model(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        hidden_sizes: int,
        n_classes: int,
        n_layers: int

    ):
        super(Model, self).__init__()
        self.embedding = nn.Embedding.from_pretrained(pretrained_embedding_weights)

        #Questao
        self.lstm_layers = nn.ModuleList()
        for i in range(len(hidden_sizes) - 1):
            self.lstm_layers.append(
                nn.LSTM(
                    input_size=hidden_sizes[i],
                    hidden_size=hidden_sizes[i + 1],
                    batch_first=True,
                )
            )

        self.fc = nn.Linear(hidden_sizes[-1], n_classes)

    def forward(self, x):
        x = self.embedding(x)
        for lstm in self.lstm_layers:
            x, _ = lstm(x)
        x = F.elu(x)
        x = x[:, -1, :]  # get the last hidden state
        logits = self.fc(x)
        return logits


### GLOVE

In [ ]:
# Assuming fasttext embeddings are available for download
from torchtext.vocab import GloVe

global_vectors = GloVe(name='6B', dim=300)
glove_weights = torch.load(f".vector_cache/glove.6B.300d.txt.pt")
pretrained_embedding_weights = glove_weights[2]

batch_size=32

model = Model(
    vocab_size=n_tokens,
    hidden_sizes=[300,300,300,300],  #[300,300,300,300]
    n_classes=n_classes,
    n_layers=1
  ).to(DEVICE)

print(model)

# Create datasets
train_dataset = VIdDataset(dataset["train"], tokenize)

# Split dataset: 80% for training, 20% for validation
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(dataset=val_subset, batch_size=batch_size, shuffle=False, drop_last=False )
train( model=model, train_loader=train_loader, val_loader=val_loader, n_epochs=100, lr=1e-3,)

testset = VIdDataset(dataset["test"], tokenize)
y_true_test, y_pred_test = evaluate_(model, testset)
trainset = VIdDataset(dataset["train"], tokenize)
y_true_train, y_pred_train = evaluate_(model, trainset)

print(f"First 10 true labels:{y_true_test[:10]}")
print(f"First 10 pred labels:{y_pred_test[:10]}")


target_names = [label_map[i] for i in range(n_classes)]
report_test = classification_report( y_true_test, y_pred_test, target_names=target_names, digits = 4)
print('Report Test:')
print(report_test)

report_train = classification_report( y_true_train, y_pred_train, target_names=target_names, digits = 4)
print('Report Train:')
print(report_train)


Model(
  (embedding): Embedding(400000, 300)
  (lstm_layers): ModuleList(
    (0-2): 3 x LSTM(300, 300, batch_first=True)
  )
  (fc): Linear(in_features=300, out_features=2, bias=True)
)


Epoch 1:   0%|          | 0/709 [00:00<?, ?it/s]

AttributeError: 'list' object has no attribute 'lower'

### FastText

In [ ]:
# Assuming fasttext embeddings are available for download
from torchtext.vocab import FastText

fasttext_vectors = FastText(language="pt")
pretrained_embedding_weights = torch.FloatTensor(fasttext_vectors.vectors)


batch_size=32

model = Model(
    vocab_size=n_tokens,
    hidden_sizes=[300,300,300, 300],  #[300,300,300]
    n_classes=n_classes,
    n_layers=1
  ).to(DEVICE)

print(model)

# Create datasets
train_dataset = VIdDataset(dataset["train"], tokenize)

# Split dataset: 80% for training, 20% for validation
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(dataset=val_subset, batch_size=batch_size, shuffle=False, drop_last=False )
train( model=model, train_loader=train_loader, val_loader=val_loader, n_epochs=100, lr=1e-3,)

testset = VIdDataset(dataset["test"], tokenize)
y_true_test, y_pred_test = evaluate_(model, testset)
trainset = VIdDataset(dataset["train"], tokenize)
y_true_train, y_pred_train = evaluate_(model, trainset)

print(f"First 10 true labels:{y_true_test[:10]}")
print(f"First 10 pred labels:{y_pred_test[:10]}")


target_names = [label_map[i] for i in range(n_classes)]
report_test = classification_report( y_true_test, y_pred_test, target_names=target_names, digits = 4)
print('Report Test:')
print(report_test)

report_train = classification_report( y_true_train, y_pred_train, target_names=target_names, digits = 4)
print('Report Train:')
print(report_train)


Model(
  (embedding): Embedding(592108, 300)
  (lstm_layers): ModuleList(
    (0-2): 3 x LSTM(300, 300, batch_first=True)
  )
  (fc): Linear(in_features=300, out_features=2, bias=True)
)


Epoch 1:   0%|          | 0/709 [00:00<?, ?it/s]

AttributeError: 'list' object has no attribute 'lower'

## Pre-Trained model (BERT)

We also tested the incorporatoin of a pre-trained model. in this case the model used was the Bert. The albertina model gave problems in the instalation.
For this model, we also used 4 layers and the early stop, but we used the patience equal to 2, because this model takes some time. We removed the original embeding from the code because the embeding is done by the Bert model, also the tokenize function is different, we need to use the tokenizer from Bert.

This model was tested using the original dataset and the extended one.

In [ ]:
class Model(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        hidden_sizes: int,
        n_classes: int,
        pad_token_id: int

    ):
        super(Model, self).__init__()
        self.lm = BertModel.from_pretrained("neuralmind/bert-base-portuguese-cased")
        for param in self.lm.parameters():
          param.requires_grad = False

        self.lstm_layers = nn.ModuleList()
        for i in range(len(hidden_sizes) - 1):
            self.lstm_layers.append(
                nn.LSTM(
                    input_size=hidden_sizes[i],
                    hidden_size=hidden_sizes[i + 1],
                    batch_first=True,
                )
            )

        self.fc = nn.Linear(hidden_sizes[-1], n_classes)
        self.pad_token_id = pad_token_id


    def forward(self, x):
      input_ids = x
      attention_mask = (input_ids != self.pad_token_id).int()
      #print(f'input_ids: {input_ids}')
      #print(f'attention_mask: {attention_mask}')
      outputs = self.lm(input_ids=input_ids, attention_mask=attention_mask)
      #print(f'Outputs: {outputs}')
      last_hidden_states = outputs['last_hidden_state']
      for lstm in self.lstm_layers:
            x, _ = lstm(last_hidden_states)
      x = F.relu(x)
      x = x[:, -1, :]
      logits = self.fc(x)
      return logits

class EarlyStopping:
    def __init__(self, patience=2, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0

    def __call__(self, val_loss):
        if self.best_score is None:
            self.best_score = val_loss
        elif val_loss > self.best_score - self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_loss
            self.counter = 0


def tokenize(text):
    tokenizer = BertTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.convert_tokens_to_ids(tokens)
    return token_ids

In [ ]:
batch_size=32
# Assuming fasttext embeddings are available for download
from torchtext.vocab import FastText

from transformers import BertModel, BertTokenizer
torch.cuda.empty_cache()
fasttext_vectors = FastText(language="pt")
pretrained_embedding_weights = torch.FloatTensor(fasttext_vectors.vectors)
print(pad_token_id)


# Initialize model with pre-trained components
model = Model(
    vocab_size=n_tokens,
    hidden_sizes=[768,768,768,768],
    n_classes=n_classes,
    pad_token_id=pad_token_id
  ).to(DEVICE)

train_dataset = VIdDataset(dataset["train"], tokenize)

# Split dataset: 80% for training, 20% for validation
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])


# Call train and evaluation functions as before
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True, drop_last=True)
val_loader = DataLoader(dataset=val_subset, batch_size=batch_size, shuffle=False, drop_last=False )
train( model=model, train_loader=train_loader, val_loader=val_loader, n_epochs=2, lr=1e-3)

testset = VIdDataset(dataset["test"], tokenize)
y_true_test, y_pred_test = evaluate_(model, testset)

trainset = VIdDataset(dataset["train"], tokenize)
y_true_train, y_pred_train = evaluate_(model, trainset)


print(f"First 10 true labels:{y_true_test[:10]}")
print(f"First 10 pred labels:{y_pred_test[:10]}")


target_names = [label_map[i] for i in range(n_classes)]
report_test = classification_report( y_true_test, y_pred_test, target_names=target_names, digits = 4)
print('Report Tests')
print(report_test)

report_train = classification_report( y_true_train, y_pred_train, target_names=target_names, digits = 4)
print('Report Train')
print(report_train)


This model lead to best F1 score.